<a href="https://colab.research.google.com/github/AlekseyZykov/Python/blob/main/%D0%A0%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%D0%9B%D0%BE%D0%B3%D0%B0%D0%BC%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Работа с логами

**Важно!**

В следующей ячейке скачивается нужный файл с логами

In [ ]:
!wget https://gist.github.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log

--2025-01-19 21:35:21--  https://gist.github.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log
Resolving gist.github.com (gist.github.com)... 140.82.112.4
Connecting to gist.github.com (gist.github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log [following]
--2025-01-19 21:35:22--  https://gist.githubusercontent.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 459418 (449K) [text/plain]
Saving to: ‘auto_purchase.log’

auto_purchase.log   100%[===================>] 448.65K  --.-KB/s    in 0.03s   

2025-01-19 21:35:22 (13.1

Файл с логами выглядит следующим образом

In [ ]:
with open('auto_purchase.log', 'r') as f:
    lines = f.readlines()

for line in lines[400:550]:
    print(line)

Задача написать функцию `count_success_and_failure`, которая принимает на вход путь к файлу с логами и подсчитывает количество успешных продлений и ошибок при списании. Функция должна вернуть кортеж из двух значений: количества успешных попыток и неуспешных.

In [ ]:

def count_success_and_failure(file_path):
  # Построчно читаем файл
  with open(file_path, 'r') as r:
    lines = r.readlines()
    # Считаем общее количество попыток
    upg_cnt = [line for line in lines if 'Обновляем подписку' in line]
    # Считаем количество ошибок обновления
    err = [line for line in lines if 'ERROR' in line]
    # Возвращаем кортеж с ответом
  return (len(upg_cnt)-len(err), len(err))


In [ ]:
count_success_and_failure('auto_purchase.log')

(1034, 186)

Задача написать функцию `auto_renewal_sub`, которая принимает на вход путь к файлу с логами и обрабатывает количество клиентов с автопродлением подписки. Мы хотим посмотреть на изменение этого показателя в динамике: посчитайте сглаженные значения с помощью метода скользящего среднего и метода медианного сглаживания.  

**Примечание:** При сглаживании берем все предыдущие значения, включая текущее, будущие значения не берем. Если в один день наблюдаем несколько записей об автопродлении - берем максимальное из имеющихся число клиентов с подпиской.

Функция должна записать в файл `auto_renewal_sub.txt` два списка, предварив их соответствущими обозначениями:

`Среднее: [2.0, 1.0, 0.67...]`

`Медиана: [2, 2, 0...]`

In [ ]:
from statistics import mean, median
def auto_renewal_sub(log_file_path):
  # Построчно читаем файл
  with open(log_file_path, 'r') as r:
    lines = r.readlines()
    # Отбираем строки с данными о кол-во автопродлений
    upg_filter = [line for line in lines if 'людей с автопродлением подписки' in line]
        # Создаем словарь с датами и максимальным значением автоподписки
  upg_dict = dict()
  for i in upg_filter:
    if upg_dict.get(i[8:18]):
      if int(i[-3:]) > upg_dict.get(i[8:18]):
        upg_dict[i[8:18]]=int(i[-3:])
    else:
      upg_dict[i[8:18]]=int(i[-3:])
  # Преобразуем словарь в список значений
  lst = list(upg_dict.values())
  avg = []
  med = []
  # Находим скользящее среднее и медиану
  for i, elem in enumerate(lst,1):
    avg.append(float(round(mean(lst[0:i]),2)))
    med.append(int(median(lst[0:i])))
  # Записываем в файл
  with open ('auto_renewal_sub.txt', 'w') as w:
    w.write(f'Среднее: {avg}\nМедиана: {med}' )
  return

In [ ]:
auto_renewal_sub('auto_purchase.log')

Напишите функцию `sub_renewal_by_day`, которая принимает на вход путь к файлу с логами и анализирует взаимосвязь дня продления подписки и количества продлений в этот день. Функция должна записать в файл `weekdays.txt` аналитическую записку в формате:

**`Количество обновлений подписки по дням недели:`**

**`Понедельник: 6`**

**`Вторник: 7`**

**`Среда: 8`**

**`...`**

In [ ]:
from datetime import datetime
from collections import Counter

def sub_renewal_by_day(file_path):
 # Читаем файл
  with open ('auto_purchase.log', 'r', encoding = 'utf-8') as f:
    reader = f.readlines()
  # Находим количество попыток продления подписки в конкретный день недели
    upd_daу = [datetime.strptime(date[8:18], '%Y-%m-%d').strftime('%A') for date in reader if 'Обновляем подписку пользователю id' in date]
    cnt_day = Counter(upd_daу)
  # Находим количество ошибок подписки в конкретный день недели
    err_day = [datetime.strptime(date[8:18], '%Y-%m-%d').strftime('%A') for date in reader if 'ERROR' in date]
    cnt_err = Counter(err_day)
  # Вычитаем из количества попыток ошибки, т.е. находим успешные попытки продления
    cnt_day.subtract(cnt_err)
  # Записываем в файл
  with open ('weekdays.txt', 'w', encoding = 'utf-8') as w:
    writer = w.write(f"""Количество обновлений подписки по дням недели:\nПонедельник: {cnt_day['Monday']}
Вторник: {cnt_day['Tuesday']}
Среда: {cnt_day['Wednesday']}
Четверг: {cnt_day['Thursday']}
Пятница: {cnt_day['Friday']}
Суббота: {cnt_day['Saturday']}
Воскресенье: {cnt_day['Sunday']}""")
  return

In [ ]:
sub_renewal_by_day('auto_purchase.log')